# 01 数据探索：信用卡欺诈数据集

这一份只看数据，不建模。从上到下一格一格运行（Shift+Enter），每格下面有"看什么"的提示。
不需要 GPU，默认的 CPU 运行环境就够。

## 1. 加载数据

Kaggle 原站下载要登录，这里用 TensorFlow 官方教程使用的公开镜像。第一次运行会下载并存一份到你的 Google Drive，之后直接从 Drive 读。
任务说明要求：用镜像时必须核对行数和欺诈条数，下一格就是核对。

In [2]:
import os
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/data-mining/creditcard.csv'
MIRROR_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'

if os.path.exists(DRIVE_PATH):
    df = pd.read_csv(DRIVE_PATH)
    print('从 Drive 读取')
else:
    df = pd.read_csv(MIRROR_URL)
    os.makedirs(os.path.dirname(DRIVE_PATH), exist_ok=True)
    df.to_csv(DRIVE_PATH, index=False)
    print('从镜像下载，并已保存到 Drive')

Mounted at /content/drive
从镜像下载，并已保存到 Drive


In [3]:
print('形状（行数, 列数）：', df.shape)
print(df['Class'].value_counts())
assert df.shape == (284807, 31), '行数或列数不对，镜像可能不完整'
assert (df['Class'] == 1).sum() == 492, '欺诈条数不对'
print('核对通过：284807 行，492 条欺诈')

形状（行数, 列数）： (284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64
核对通过：284807 行，492 条欺诈


## 中文字体设置

Colab 默认字体不带中文，图上的中文会变成方框。这一格装一个中文字体，之后所有图都能显示中文。只需要在每次打开笔记本时运行一次。

In [ ]:
!apt-get -qq install -y fonts-noto-cjk > /dev/null
import glob
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = glob.glob('/usr/share/fonts/**/NotoSansCJK*-Regular.ttc', recursive=True)[0]
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['axes.unicode_minus'] = False
print('中文字体已设置：', plt.rcParams['font.family'])

## 2. 看几行样例

看什么：每一行是一笔交易。Time 是距第一笔交易的秒数，V1~V28 是脱敏特征，Amount 是金额，Class 是标签（0 正常，1 欺诈）。

In [4]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## 3. 每一列的基本统计

看什么：
- V1~V28 的均值几乎都是 0，这是 PCA 处理留下的痕迹。
- Time 和 Amount 的数值量级比 V 列大得多，这就是后面要做标准化的原因。

In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Time,284807.0,9.481386e+04,47488.145955,0.000000,54201.500000,84692.000000,139320.500000,172792.000000
V1,284807.0,1.168375e-15,1.958696,-56.407510,-0.920373,0.018109,1.315642,2.454930
V2,284807.0,3.416908e-16,1.651309,-72.715728,-0.598550,0.065486,0.803724,22.057729
V3,284807.0,-1.379537e-15,1.516255,-48.325589,-0.890365,0.179846,1.027196,9.382558
V4,284807.0,2.074095e-15,1.415869,-5.683171,-0.848640,-0.019847,0.743341,16.875344
V5,284807.0,9.604066e-16,1.380247,-113.743307,-0.691597,-0.054336,0.611926,34.801666
V6,284807.0,1.487313e-15,1.332271,-26.160506,-0.768296,-0.274187,0.398565,73.301626
V7,284807.0,-5.556467e-16,1.237094,-43.557242,-0.554076,0.040103,0.570436,120.589494
V8,284807.0,1.213481e-16,1.194353,-73.216718,-0.208630,0.022358,0.327346,20.007208
V9,284807.0,-2.406331e-15,1.098632,-13.434066,-0.643098,-0.051429,0.597139,15.594995


## 4. 缺失值和重复行

看什么：缺失值是不是 0（报告"缺失值处理"一栏的依据）；有没有完全相同的重复行，有的话后面要决定删不删。

In [6]:
print('缺失值总数：', df.isnull().sum().sum())
dup = df.duplicated()
print('完全重复的行数：', dup.sum())
print('重复行里的欺诈条数：', df.loc[dup, 'Class'].sum())

缺失值总数： 0
完全重复的行数： 1081
重复行里的欺诈条数： 19


## 5. 类别到底有多不平衡

In [7]:
counts = df['Class'].value_counts()
print(f'正常 : 欺诈 = {counts[0]} : {counts[1]}，约 {counts[0] / counts[1]:.0f} : 1')
print(f'欺诈占比：{counts[1] / len(df):.4%}')
print(f'如果模型全部猜"正常"，准确率 = {counts[0] / len(df):.4%}')

正常 : 欺诈 = 284315 : 492，约 578 : 1
欺诈占比：0.1727%
如果模型全部猜"正常"，准确率 = 99.8273%


## 6. 金额 Amount

**左图（箱线图）**：横轴是类别（0 正常 / 1 欺诈），纵轴是交易金额。箱子外面的圈就是离群点。注意欺诈交易里也有不少离群点，想想为什么异常值不能随手删。

**右图（直方图）**：横轴是 log(1+金额)，把极端大的金额压扁方便看；纵轴是密度（各自占本类的比例）。两类样本数差了 578 倍，直接数个数的话欺诈那一类会矮到看不见，所以各自换算成比例再比形状。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print(df.groupby('Class')['Amount'].describe())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df.boxplot(column='Amount', by='Class', ax=axes[0])
axes[0].set_title('各类别的交易金额')
axes[0].set_xlabel('类别（0 正常 / 1 欺诈）')
axes[0].set_ylabel('交易金额')

for c, name in [(0, '正常'), (1, '欺诈')]:
    axes[1].hist(np.log1p(df.loc[df['Class'] == c, 'Amount']), bins=50,
                 density=True, alpha=0.5, label=name)
axes[1].set_title('金额分布（取对数后）')
axes[1].set_xlabel('log(1 + 金额)')
axes[1].set_ylabel('密度（占本类的比例）')
axes[1].legend()
plt.suptitle('')
plt.tight_layout()
plt.show()

## 7. 时间 Time

数据跨两天，这里把秒数换算成"第几个小时"（0~23）。起点是第一笔交易，不一定是午夜，所以这个小时是相对时间。

**横轴**：相对小时。**灰色柱子（左纵轴）**：这个小时里有多少笔交易。**红线（右纵轴）**：这个小时里欺诈交易占的百分比。

看什么：交易量有没有昼夜起伏；交易少的时段，欺诈比例是不是反而更高。

In [ ]:
hour = (df['Time'] // 3600) % 24
by_hour = df.groupby(hour)['Class'].agg(['count', 'mean'])

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.bar(by_hour.index, by_hour['count'], color='lightgray', label='交易笔数')
ax1.set_xlabel('相对小时（0~23）')
ax1.set_ylabel('交易笔数')
ax2 = ax1.twinx()
ax2.plot(by_hour.index, by_hour['mean'] * 100, color='red', marker='o', label='欺诈比例')
ax2.set_ylabel('欺诈比例（%）')
fig.legend(loc='upper right')
plt.title('每小时的交易笔数和欺诈比例')
plt.tight_layout()
plt.show()

## 8. 哪些 V 特征最能区分两类

对每个 V 特征，算"欺诈均值 − 正常均值"，再除以这一列的标准差，数值越大说明两类在这个特征上差得越远。除以标准差是为了让不同特征可以放在一起比：有的特征本身波动就大，均值差 2 不算什么；有的特征波动很小，均值差 2 就很显著。

这只是快速看一眼的方法之一，不是唯一答案。正式做特征选择时还可以用相关系数、随机森林的特征重要性等方法，结果可能不完全一样。

**图**：画出差别最大的 4 个特征。每张小图的横轴是该特征的取值，纵轴是密度（占本类的比例）。看两类的分布是明显错开，还是大部分重叠。

In [ ]:
feats = [f'V{i}' for i in range(1, 29)]
means = df.groupby('Class')[feats].mean()
gap = ((means.loc[1] - means.loc[0]) / df[feats].std()).abs().sort_values(ascending=False)
print('两类差别最大的 10 个特征：')
print(gap.head(10).round(2))

top4 = gap.index[:4]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, f in zip(axes, top4):
    for c, name in [(0, '正常'), (1, '欺诈')]:
        ax.hist(df.loc[df['Class'] == c, f], bins=60, density=True, alpha=0.5, label=name)
    ax.set_title(f)
    ax.set_xlabel(f'{f} 的取值')
axes[0].set_ylabel('密度（占本类的比例）')
axes[0].legend()
plt.tight_layout()
plt.show()

## 9. 特征之间的相关性

**左图（热力图）**：横轴和纵轴都是 V1~V28，每个格子的颜色是这两个特征的相关系数，红色正相关、蓝色负相关、白色不相关。看什么：除了对角线（自己和自己，必然是 1），几乎全是白色，这是 PCA 的性质——主成分之间互不相关。

**右图（条形图）**：纵轴是每个特征，横轴是它和 Class 的相关系数，越往两边说明和欺诈的关系越强。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im = axes[0].imshow(df[feats].corr(), cmap='coolwarm', vmin=-1, vmax=1)
axes[0].set_xticks(range(28))
axes[0].set_xticklabels(feats, rotation=90, fontsize=7)
axes[0].set_yticks(range(28))
axes[0].set_yticklabels(feats, fontsize=7)
axes[0].set_title('V1~V28 两两之间的相关系数')
plt.colorbar(im, ax=axes[0], label='相关系数')

corr_class = df[feats + ['Time', 'Amount']].corrwith(df['Class']).sort_values()
axes[1].barh(corr_class.index, corr_class.values)
axes[1].set_title('各特征与 Class 的相关系数')
axes[1].set_xlabel('相关系数')
axes[1].set_ylabel('特征')
axes[1].tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.show()

## 10.（选做）把高维数据压到二维看一眼

用 t-SNE 把全部 492 条欺诈 + 随机 3000 条正常交易投影到平面上，大约要跑 1 分钟。

**横纵轴**：t-SNE 算出来的两个坐标，本身没有具体含义，只看点和点的相对远近：离得近的点，在原来 28 维空间里也比较像。另外，簇的大小和簇之间的距离都不可靠，换一个随机种子，图的样子可能会变。

看什么：欺诈点是聚成一团，还是分成好几簇——这对应你之前"隐空间里有细分"的猜测。这是原始特征的投影，不是神经网络的隐空间，只能当初步线索，不能当结论。

In [ ]:
from sklearn.manifold import TSNE

sample = pd.concat([
    df[df['Class'] == 1],
    df[df['Class'] == 0].sample(3000, random_state=42),
])
emb = TSNE(n_components=2, random_state=42).fit_transform(sample[feats])

plt.figure(figsize=(7, 6))
for c, name, color in [(0, '正常', 'lightgray'), (1, '欺诈', 'red')]:
    m = (sample['Class'] == c).values
    plt.scatter(emb[m, 0], emb[m, 1], s=6, c=color, label=name, alpha=0.7)
plt.xlabel('t-SNE 坐标 1（无具体含义）')
plt.ylabel('t-SNE 坐标 2（无具体含义）')
plt.legend()
plt.title('t-SNE 投影：全部欺诈 + 3000 条正常交易')
plt.show()

## 看完之后记下来（写报告要用，用你自己的话）

1. 缺失值有多少？重复行有多少，里面有没有欺诈？
2. 不平衡比例是多少？全猜"正常"的准确率是多少？
3. 欺诈交易的金额和正常交易比，有什么不同？异常值该不该删？
4. 哪几个 V 特征最能区分两类？两类的分布是错开还是重叠？
5. t-SNE 图里，欺诈是一团还是几团？